In [1]:
import joblib
import pandas as pd

# Load the trained XGBoost pipeline
model = joblib.load("traffic_binary_lr_pipeline.pkl")

In [2]:
# New raw input sample
sample = pd.DataFrame([
    {
        "road": "third mainland bridge",
        "fixed_corridor": "oworonshole – adekunle",
        "day": "Friday",
        "weather": "partly cloudy",
        "temperature": 28.0,
        "rain_chance": 0.25,
        "traffic_pattern": "G",
        "congestion_location": "none evident",
        "obs_hour": 16,
    }
])

In [3]:
# Validate sample against the features the pipeline was trained on
preprocessor = model.named_steps["prep"]
expected = list(preprocessor.feature_names_in_)

print("Expected features:", expected)

missing = [c for c in expected if c not in sample.columns]
extra = [c for c in sample.columns if c not in expected]

assert not missing, f"Sample is missing required columns: {missing}"
if extra:
    print(f"Ignoring extra columns not seen in training: {extra}")

# Reorder columns to match training exactly
sample = sample[expected]
print("Sample columns OK.")

Expected features: ['road', 'fixed_corridor', 'day', 'weather', 'temperature', 'rain_chance', 'obs_hour']
Ignoring extra columns not seen in training: ['traffic_pattern', 'congestion_location']
Sample columns OK.


In [4]:
# Predict traffic status (0 = Normal, 1 = Congested)
prediction = model.predict(sample)[0]
proba = model.predict_proba(sample)[0]

status = "🚨 Congested Flow" if prediction == 1 else "✅ Normal Flow"

print(f"Prediction: {status}")
print(f"P(Normal) = {proba[0]:.2f} | P(Congested) = {proba[1]:.2f}")

Prediction: 🚨 Congested Flow
P(Normal) = 0.10 | P(Congested) = 0.90


In [5]:
# Sanity check: compare the model against the rush-hour rule discovered in EDA.
# Training data was deterministic outside transition hours:
#   congested 100%: hours 7-9, 15-19
#   normal    100%: hours 6, 10, 11, 21, 22
#   ~50/50        : hours 12-14, 20  (irreducible noise)

def rush_hour_rule(hour: int):
    if hour in (7, 8, 9, 15, 16, 17, 18, 19):
        return 1
    if hour in (6, 10, 11, 21, 22):
        return 0
    return None  # transition hour

h = int(sample["obs_hour"].iloc[0])
rule = rush_hour_rule(h)

rule_str = ("🚨 Congested" if rule == 1
            else "✅ Normal" if rule == 0
            else "⚠️ ~50/50 (transition hour, either class is plausible)")
print(f"obs_hour = {h}  →  rule says: {rule_str}")

if rule is None:
    print("ℹ️ Transition hour — trust the probabilities, not the hard label.")
elif rule != prediction:
    print("⚠️ WARNING: model disagrees with the deterministic rush-hour rule.")
    print("The loaded pickle is likely STALE (saved before obs_hour was added to X).")
    print("Retrain the pipeline with obs_hour included, then re-save:")
    print('    joblib.dump(xgb_pipe, "traffic_binary_xgb_pipeline.pkl")')
else:
    print("✅ Model agrees with the rush-hour rule.")

obs_hour = 16  →  rule says: 🚨 Congested
✅ Model agrees with the rush-hour rule.
